In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 1. Paths
# =========================
BASE_DIR = Path(r"C:\Projet_filrouge\oncobio_decision_analytics\01_Data")
INTERIM_DIR = BASE_DIR / "Interim"
PROCESSED_DIR = BASE_DIR / "Processed"
EXTERNAL_DIR = BASE_DIR / "External_api"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# =========================
# 2. Load sources
# =========================
breast = pd.read_csv(INTERIM_DIR / "SEER Breast Cancer Dataset .csv")
lung = pd.read_csv(PROCESSED_DIR / "ncctg_lung_patient_1000_sdv.csv")
biomarkers = pd.read_csv(EXTERNAL_DIR / "biomarker_reference.csv")

print("Breast:", breast.shape)
print("Lung:", lung.shape)
print("Biomarkers:", biomarkers.shape)

Breast: (4024, 16)
Lung: (1228, 9)
Biomarkers: (10, 5)


In [3]:
# =========================
# 3. Prepare breast dataset
# =========================
breast = breast.rename(columns={
    "Age": "age",
    "Survival Months": "survival_months",
    "Status": "event",
    "6th Stage": "stage"
})

breast = breast.dropna(subset=["age", "survival_months"]).copy()

breast["event"] = breast["event"].map({
    "Alive": 0,
    "Dead": 1
})

if "sex" not in breast.columns:
    breast["sex"] = "F"

if "stage" not in breast.columns:
    breast["stage"] = np.nan

breast["ecog"] = np.nan
breast["cancer_type"] = "Breast"

breast_prepared = breast[[
    "age", "sex", "survival_months", "event", "stage", "ecog", "cancer_type"
]].copy()

In [4]:
# =========================
# 4. Prepare lung dataset
# =========================
lung = lung.rename(columns={
    "TIME": "survival_months",
    "Y": "event",
    "ECOG": "ecog",
    "SEX": "sex",
    "AGE": "age",
    "STAGE": "stage"
})

# si certaines colonnes sont déjà minuscules, ce bloc évite les erreurs
lung.columns = [c.strip() for c in lung.columns]

for col in ["age", "sex", "survival_months", "event"]:
    if col not in lung.columns:
        raise ValueError(f"Colonne manquante dans lung: {col}")

lung["survival_months"] = pd.to_numeric(lung["survival_months"], errors="coerce")
lung["age"] = pd.to_numeric(lung["age"], errors="coerce")
lung["event"] = pd.to_numeric(lung["event"], errors="coerce")

lung = lung.dropna(subset=["age", "survival_months", "event"]).copy()
lung["survival_months"] = lung["survival_months"].clip(lower=1)
lung["event"] = lung["event"].astype(int)

if "stage" not in lung.columns:
    lung["stage"] = np.nan

if "ecog" not in lung.columns:
    lung["ecog"] = np.nan

# harmonisation sexe
lung["sex"] = lung["sex"].replace({
    1: "M",
    2: "F",
    "1": "M",
    "2": "F",
    "male": "M",
    "female": "F",
    "Male": "M",
    "Female": "F"
})

lung["cancer_type"] = "Lung"

lung_prepared = lung[[
    "age", "sex", "survival_months", "event", "stage", "ecog", "cancer_type"
]].copy()

In [5]:
# =========================
# 5. Build patients_master
# =========================
patients_master = pd.concat(
    [breast_prepared, lung_prepared],
    ignore_index=True
)

patients_master.insert(0, "patient_id", range(1, len(patients_master) + 1))

patients_master["risk_group"] = pd.cut(
    patients_master["survival_months"],
    bins=[0, 12, 36, 1200],
    labels=["High Risk", "Medium Risk", "Low Risk"]
)

In [6]:
# =========================
# 6. KPI summary
# =========================
kpi_summary = (
    patients_master.groupby("cancer_type")
    .agg(
        total_patients=("patient_id", "count"),
        mean_age=("age", "mean"),
        death_rate=("event", "mean"),
        mean_survival=("survival_months", "mean"),
        median_survival=("survival_months", "median")
    )
    .reset_index()
)

kpi_summary["death_rate"] = (kpi_summary["death_rate"] * 100).round(2)
kpi_summary["mean_age"] = kpi_summary["mean_age"].round(1)
kpi_summary["mean_survival"] = kpi_summary["mean_survival"].round(1)
kpi_summary["median_survival"] = kpi_summary["median_survival"].round(1)

# =========================
# 7. Survival summary
# =========================
survival_summary = (
    patients_master.groupby(["cancer_type", "risk_group"], observed=False)
    .agg(
        total_patients=("patient_id", "count"),
        death_rate=("event", "mean"),
        median_survival=("survival_months", "median")
    )
    .reset_index()
)

survival_summary["death_rate"] = (survival_summary["death_rate"] * 100).round(2)


In [7]:
# =========================
# 8. Patient-biomarker bridge
# =========================
# Ici, comme on n’a pas encore de vraie liaison patient ↔ biomarqueur mesurée,
# on crée un bridge simple "project-level" pour permettre la modélisation SQL.
# Plus tard, vous pourrez le remplacer par une vraie table clinique si vous avez l’expression génique.

biomarkers = biomarkers.copy()

# harmonisation colonnes biomarqueurs
biomarkers.columns = [c.strip().lower() for c in biomarkers.columns]

rename_map = {}
if "symbol" in biomarkers.columns:
    rename_map["symbol"] = "gene_symbol"
if "geneid" in biomarkers.columns:
    rename_map["geneid"] = "gene_id"
if "official_name" not in biomarkers.columns and "name" in biomarkers.columns:
    rename_map["name"] = "official_name"

biomarkers = biomarkers.rename(columns=rename_map)

for col in ["gene_symbol", "gene_id"]:
    if col not in biomarkers.columns:
        biomarkers[col] = np.nan

if "official_name" not in biomarkers.columns:
    biomarkers["official_name"] = np.nan

if "description" not in biomarkers.columns:
    biomarkers["description"] = np.nan

biomarkers = biomarkers[["gene_symbol", "gene_id", "official_name", "description"]].drop_duplicates().reset_index(drop=True)
biomarkers.insert(0, "biomarker_id", range(1, len(biomarkers) + 1))

# bridge simple : associer tous les biomarqueurs à chaque type de cancer
bridge_rows = []
for _, patient in patients_master[["patient_id", "cancer_type"]].iterrows():
    for _, biom in biomarkers[["biomarker_id", "gene_symbol"]].iterrows():
        bridge_rows.append({
            "patient_id": patient["patient_id"],
            "biomarker_id": biom["biomarker_id"],
            "gene_symbol": biom["gene_symbol"],
            "association_source": "project_reference"
        })

patient_biomarker_bridge = pd.DataFrame(bridge_rows)

# =========================
# 9. Export final files
# =========================
breast_prepared.to_csv(PROCESSED_DIR / "breast_prepared.csv", index=False)
lung_prepared.to_csv(PROCESSED_DIR / "lung_prepared.csv", index=False)
patients_master.to_csv(PROCESSED_DIR / "patients_master.csv", index=False)
biomarkers.to_csv(PROCESSED_DIR / "biomarker_reference_clean.csv", index=False)
kpi_summary.to_csv(PROCESSED_DIR / "kpi_summary.csv", index=False)
survival_summary.to_csv(PROCESSED_DIR / "survival_summary.csv", index=False)
patient_biomarker_bridge.to_csv(PROCESSED_DIR / "patient_biomarker_bridge.csv", index=False)

print("Fichiers exportés avec succès dans :", PROCESSED_DIR)
print(list(PROCESSED_DIR.glob("*.csv")))

Fichiers exportés avec succès dans : C:\Projet_filrouge\oncobio_decision_analytics\01_Data\Processed
[WindowsPath('C:/Projet_filrouge/oncobio_decision_analytics/01_Data/Processed/biomarker_reference_clean.csv'), WindowsPath('C:/Projet_filrouge/oncobio_decision_analytics/01_Data/Processed/breast_prepared.csv'), WindowsPath('C:/Projet_filrouge/oncobio_decision_analytics/01_Data/Processed/kpi_summary.csv'), WindowsPath('C:/Projet_filrouge/oncobio_decision_analytics/01_Data/Processed/lung_cancer_clean.csv'), WindowsPath('C:/Projet_filrouge/oncobio_decision_analytics/01_Data/Processed/lung_prepared.csv'), WindowsPath('C:/Projet_filrouge/oncobio_decision_analytics/01_Data/Processed/ncctg_lung_patient_1000_sdv.csv'), WindowsPath('C:/Projet_filrouge/oncobio_decision_analytics/01_Data/Processed/patients_master.csv'), WindowsPath('C:/Projet_filrouge/oncobio_decision_analytics/01_Data/Processed/patient_biomarker_bridge.csv'), WindowsPath('C:/Projet_filrouge/oncobio_decision_analytics/01_Data/Proc